# The answer step

An open LLM (Qwen3-4B-Instruct, Apache-2.0) reads the top 3 articles for each ticket, judges whether they contain the answer, and drafts replies with citations. Its judgement is then added to the resolve / assist / escalate decision. Needs a **T4 GPU**; about an hour, plus 35 minutes if retrieval results must be rebuilt.

In [ ]:
REPO_URL = 'https://github.com/rushitpatel100-oss/ticket-triage-ai.git'
import os
if not os.path.exists('/content/ticket-triage-ai'):
    !git clone -q $REPO_URL /content/ticket-triage-ai
%cd /content/ticket-triage-ai
!git pull -q
!pip install -q 'sentence-transformers>=3.0' 'nltk>=3.8' 'transformers>=4.51'

In [ ]:
import os
if not os.path.exists('data/processed/retrieval/stackexchange_runs.parquet'):
    !python -m src.sources --only techqa stackexchange
    !python -m src.retrieval --datasets techqa --rerankers cross-encoder/ms-marco-MiniLM-L-6-v2 BAAI/bge-reranker-base --fp16
    !python -m src.retrieval --datasets stackexchange --rerankers cross-encoder/ms-marco-MiniLM-L-6-v2 BAAI/bge-reranker-base --fp16 --max-queries-per-split 2000
else:
    print('Using existing retrieval results')

In [ ]:
# Smoke test first: 32 questions, 4 answers (a couple of minutes)
!python -m src.answer --datasets techqa --limit 32 --max-answers 4 | tail -5

In [ ]:
# Full run, one dataset at a time (results are merged): every question with retrieval results
# is judged; 150 test answers per dataset; the red-team set runs once
!python -m src.answer --datasets techqa
!python -m src.answer --datasets stackexchange --skip-redteam

In [ ]:
# Decision with the LLM judgement as an extra signal
!python -m src.decision --llm-judge